# Vault SCEP: validación por CLI sin Intune

Objetivo: comparar challenge compartido, transición a cert auth y cert auth desde la primera petición SCEP sin challenge, midiendo identidades y contadores en cada fase.

**Resultado observado en Vault Enterprise 2.2.0:** tres altas con challenge generaron tres Entity clients. Renovar con el mismo challenge añadió cero. Pasar los tres equipos al mount cert generó otras tres entidades, aunque mantuvieron CN/SAN. Una segunda renovación mediante cert mantuvo su entidad y añadió cero. No se enlazaron ni fusionaron aliases entre métodos.

**Nuevo caso cert-only validado (20261028T165249Z):** tres dispositivos con certificados individuales preaprovisionados mediante API, nueve peticiones SCEP sin challenge y dos rondas de renovación. Contador de clientes: 29 → 32 → 32 → 32. Cada equipo mantuvo su entidad. La primera petición SCEP ya es RenewalReq porque el equipo posee un certificado previo de la CA.

El código de las celdas es Bash mediante `%%bash`, compatible con el kernel habitual del repositorio. No se utiliza código Python para configurar Vault, generar certificados, enviar SCEP ni analizar resultados. Dependencias: Bash, Vault CLI, curl, OpenSSL 3, jq, Go y kubectl (para leer la auditoría del laboratorio).

## 1. Preflight

Ejecutar desde la raíz del repositorio. El laboratorio carga `/tmp/vault/config.env` y utiliza el namespace root y el contexto Kubernetes `workshop-k8s` (sobrescribible con `KUBE_CONTEXT`). No imprime tokens ni challenges.

In [ ]:
%%bash
if [[ -f /tmp/vault/config.env ]]; then
  set -a
  source /tmp/vault/config.env
  set +a
fi
set -euo pipefail
unset VAULT_NAMESPACE
for tool in bash vault curl jq go kubectl; do command -v "$tool"; done
vault status -format=json | jq '{version,sealed,cluster_name}'
bash -n scripts/test_scep_shared_credential.sh


## 2. Ejecutar challenge y cert auth

La siguiente celda crea una CA de prueba de 24 horas, dos auth mounts dedicados (scep y cert), una política limitada a la ruta SCEP del laboratorio y un dispositivo de auditoría, todos con nombres únicos. Ambos autenticadores emiten batch tokens. El rol cert confía únicamente en la CA del laboratorio y restringe CN y DNS SAN a device-*.scep-cli.example.test. No se cambia el listener TLS ni se instalan componentes en Excel.

Secuencia: GetCACaps/GetCACert; tres PKCSReq con challenge; un RenewalReq con el mismo challenge; un control con challenge incorrecto; tres RenewalReq sin challenge, firmados por los certificados de alta; otra renovación firmada por el certificado ya renovado de device-01; dos controles negativos: firmante no confiable y firmante confiable con challenge incorrecto. Cada solicitud válida usa una nueva clave/CSR con el mismo CN/SAN.

En cert auth, el certificado y su clave firman el mensaje CMS SCEP. curl no envía token Vault, usuario/contraseña ni certificado TLS cliente. El CSR omite completamente challengePassword y se comprueba su ausencia. La auditoría debe mostrar el login delegado bajo el mount cert. Con challenge presente, debe mostrar el mount scep.

Todas las peticiones HTTP utilizan curl con verificación TLS. El auxiliar Go de `scripts/scep-message/` construye el PKCS#7 con SHA-256 y AES-256-CBC y verifica firma, CA, nonce, transactionID y certificado recibido. HTTP 200 por sí solo no significa éxito: se comprueba también pkiStatus.

La ejecución elimina sus mounts, política, dispositivo de auditoría, challenge y claves privadas. Conserva certificados públicos, mensajes SCEP cifrados y evidencia redactada en la ruta que imprime. Cada nueva ejecución añade uso histórico: ocho hojas y una CA, y en la configuración probada seis clientes. No modifica los recursos Intune anteriores. Para repetir solamente el baseline anterior: SCEP_TEST_CERT_AUTH=false bash scripts/test_scep_shared_credential.sh.

In [ ]:
%%bash
set -euo pipefail
SCEP_TEST_CERT_AUTH=true bash scripts/test_scep_shared_credential.sh


## 3. Inspeccionar evidencia

Las celdas siguientes consultan la ejecución real 20261028T140639Z. Para otra ejecución, sustituye `EVIDENCE` por la ruta que imprima el script. Las celdas no se marcan como ejecutadas artificialmente; los resultados medidos están en los archivos de evidencia.

| Fase | Entity clients | Non-entity clients |
|---|---:|---:|
| Antes | 11 | 0 |
| Después de tres altas con challenge | 14 | 0 |
| Después de renovar con challenge | 14 | 0 |
| Después de renovar tres equipos mediante cert | 17 | 0 |
| Después de repetir renovación mediante cert | 17 | 0 |
| Después de los controles negativos | 17 | 0 |

Los tres CN tienen aliases bajo auth_scep_49ed9c61 y otros tres bajo auth_cert_b5d1ee13. Cada par de aliases corresponde a entidades diferentes, no enlazadas. Ocho seriales y fingerprints distintos. El contador PKI pasó de 32 a 41: una CA y ocho hojas. El incremento de clientes se debe al cambio de identidad autenticada, no al hecho de emitir otro certificado.

In [ ]:
%%bash
set -euo pipefail
EVIDENCE=evidence/scep-auth-methods-20261028T140639Z
jq . "$EVIDENCE/summary.json"
jq . "$EVIDENCE/cert-auth-comparison.json"
jq . "$EVIDENCE/entity-aliases.json"
jq . "$EVIDENCE/authentication-routing.json"
jq '.[] | {step,request_id,client_id,entity_id}' "$EVIDENCE/operations.json"


In [ ]:
%%bash
set -euo pipefail
EVIDENCE=evidence/scep-auth-methods-20261028T140639Z
jq '.certificates[] | {common_name,serial,sha256,pki_status,signature_verified,nonce_verified}' "$EVIDENCE/certificate-results.json"
jq . "$EVIDENCE/wrong-challenge-result.json"
jq . "$EVIDENCE/untrusted-cert-result.json"
jq . "$EVIDENCE/trusted-cert-wrong-challenge-result.json"
jq . "$EVIDENCE/activity-export-status.json"


## 4. Interpretación y referencias

La prueba valida SCEP con challenge estático compartido, transición a renovación sin challenge mediante cert auth y el flujo cert-only desde la primera petición SCEP (sección 5). No valida el challenge externo de Intune ni el enlace de aliases entre métodos. Cambiar de autenticador sin enlazar identidades puede añadir un cliente por dispositivo en la transición. Las renovaciones siguientes con la misma identidad no añadieron clientes. Este resultado técnico no sustituye las condiciones de licencia del contrato.

Los batch tokens observados tienen entidades, por lo que no se aplica la deduplicación de non-entity tokens por namespace/políticas. El export detallado puede tardar en publicar las filas; se guardan también auditoría, aliases y contadores entre fases. Ningún control negativo emitió certificado ni añadió clientes. No se han desactivado validaciones de firmas, confianza o TLS.

- [Informe y seriales completos](PKI_EST_SCEP_Validation_Report.md#scep-without-intune-curl-validation)
- [Resultado de la transición a cert auth](PKI_EST_SCEP_Validation_Report.md#scep-cert-auth-renewal-without-challenge)
- [Cert-only desde la primera petición y tabla resumen](PKI_EST_SCEP_Validation_Report.md#scep-cert-only-from-the-first-request)
- [Script ejecutable](scripts/test_scep_shared_credential.sh)
- [Configuración SCEP de Vault](https://developer.hashicorp.com/vault/docs/secrets/pki/scep)
- [API de autenticación SCEP](https://developer.hashicorp.com/vault/api-docs/auth/scep)
- [API de autenticación cert](https://developer.hashicorp.com/vault/api-docs/auth/cert)
- [RFC 8894](https://www.rfc-editor.org/rfc/rfc8894.html)


## 5. Cert auth desde la primera petición SCEP, sin challenge

Validado en la ejecución **20261028T165249Z**. Tres equipos tienen certificados/clave individuales previos, emitidos por la API administrativa bajo la misma CA de laboratorio. No se usa challenge ni se crea un auth mount de tipo scep. Se configura únicamente cert auth, con un único accessor y el mismo rol, política y namespace durante todo el ciclo.

La primera petición SCEP de cada equipo se firma con su certificado previo. La primera renovación se firma con el certificado devuelto por SCEP; la segunda, con el certificado de la renovación anterior. Todas son **RenewalReq (messageType=17)** porque existe una credencial de la CA desde el inicio. No se afirma que un dispositivo sin certificado previo pueda obtener el primero de esta forma. No se ha probado una CA externa.

| Fase | Peticiones correctas | Entity clients | Clientes adicionales |
|---|---:|---:|---:|
| Antes | — | 29 | — |
| Tres certificados previos mediante API administrativa | — | 29 | 0 |
| Primera petición SCEP de cada equipo | 3 | 32 | 3 |
| Primera ronda de renovación | 3 | 32 | 0 |
| Segunda ronda de renovación | 3 | 32 | 0 |
| Control con firmante no confiable | 0 | 32 | 0 |

Las nueve peticiones válidas devolvieron HTTP 200 y pkiStatus=0. El control negativo devolvió HTTP 200 y pkiStatus=2, sin emitir certificado ni añadir cliente. Los tres entity_id permanecieron estables; cada alias es el CN del dispositivo bajo auth_cert_093c3e4d. No se enlazaron aliases. El contador de certificados pasó de 83 a 96: una CA, tres hojas previas y nueve hojas SCEP.

Se verificaron las firmas CMS, confianza en la CA, nonce y transactionID, la ausencia de challenge y la correspondencia exacta entre el firmante de cada renovación y el certificado emitido en el paso anterior. Una verificación independiente con OpenSSL comprobó también solicitudes, respuestas, cadenas, firmas CSR, claves públicas y Subject/SAN. Los nueve seriales y claves CSR son distintos.

El export detallado seguía pendiente de propagación al muestrearlo; los resultados se apoyan en auditoría, Identity API y contadores por fase. Se eliminaron mounts, política, dispositivo de auditoría y claves privadas temporales; se conservaron evidencia pública/cifrada, identidades residuales, auditoría HMAC del servidor y uso histórico.

La celda siguiente **no ejecuta nuevas emisiones por defecto**. Cambiar RUN_SCEP_CERT_ONLY a true repite el laboratorio y añade uso histórico. La compilación del auxiliar Go utiliza solamente dependencias en caché (GOPROXY=off y GOSUMDB=off); no instala nada. Las dos celdas posteriores solo leen la evidencia existente.


In [ ]:
%%bash
set -euo pipefail
RUN_SCEP_CERT_ONLY=false
if [[ "$RUN_SCEP_CERT_ONLY" == true ]]; then
  bash scripts/test_scep_shared_credential.sh --cert-only
else
  printf '%s\n' 'No new requests sent. Read the saved evidence below.'
fi


In [ ]:
%%bash
set -euo pipefail
EVIDENCE=evidence/scep-cert-only-20261028T165249Z
jq -e -f scripts/verify_scep_cert_only.jq "$EVIDENCE/lifecycle-operations.json"
jq '{status,run_id,authentication,preprovisioned_certificates,first_request_message_type,enrollments,renewals,unique_leaf_serials,untrusted_signer_rejected,clients,certificates}' "$EVIDENCE/summary.json"
jq '.[] | {phase,entity_clients:.clients.entity_clients,delta}' "$EVIDENCE/phase-deltas.json"


In [ ]:
%%bash
set -euo pipefail
EVIDENCE=evidence/scep-cert-only-20261028T165249Z
jq . "$EVIDENCE/entity-aliases.json"
jq '.[] | {step,request_id,entity_id,auth_paths,challenge_present,signer_reference,signer_serial,issued_serial:.result.serial}' "$EVIDENCE/lifecycle-operations.json"
jq . "$EVIDENCE/untrusted-cert-result.json"
jq . "$EVIDENCE/activity-export-status.json"
jq -e '.remaining | length == 0' "$EVIDENCE"/cleanup-*.json
rg 'Subject/SAN PASS' "$EVIDENCE/independent-verification.txt"
